# Video event detection + temporal localization (VideoMAEv2-S, small data)

**Pipeline**
1. Download the dataset (Google Drive) and the pretrained **VideoMAEv2 ViT-Small** (distilled from ViT-g, Kinetics-710).
2. Decode every 8 s clip once at 8 FPS / 224 px and cache it.
3. Train a 2-output multi-label classifier (`event_1`, `event_2`) with **multiple-instance learning**:
   each clip gives K 2-second windows; the clip label supervises a smooth-max (LogSumExp) over the windows,
   so the model learns *which* window contains the event without needing timestamps.
4. 5-fold cross-validation (split by clip / group) -> out-of-fold metrics + calibrated thresholds.
5. Train the final model on all data and export a small fp16 checkpoint (~45 MB).
6. Sliding-window inference + segment merging -> `Detected X from a s until b s`.

Training: RTX 6000 (48 GB) / A100. Inference: any consumer GPU (RTX 4070 is plenty) via `infer.py`.

## 0. Environment

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "gdown>=5.1" huggingface_hub opencv-python-headless scikit-learn matplotlib
WORK = "/content/work" if IN_COLAB else os.path.abspath("work")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
os.makedirs("ved", exist_ok=True)
print("working dir:", os.getcwd())

## 1. Write the code package (`ved/`) and `infer.py`

In [ ]:
%%writefile ved/__init__.py
"""Video event detection & temporal localization (VideoMAEv2 + MIL)."""

In [ ]:
%%writefile ved/labels.py
"""Expose stable event identifiers without copying checkpoint label text."""
from __future__ import annotations


def neutral_metadata(meta: dict, num_classes: int) -> dict:
    """Keep output order and calibrated thresholds from an existing export.

    Only inference settings are retained. Historical training configuration and
    display text are intentionally omitted from the returned metadata.
    """
    previous = list(meta["class_names"])
    if num_classes < 1 or len(previous) != num_classes or len(set(previous)) != num_classes:
        raise ValueError("Checkpoint class metadata does not match the output head.")
    names = [f"event_{i + 1}" for i in range(num_classes)]
    thresholds = meta.get("thresholds", {})
    return {
        "class_names": names,
        "display_names": {name: f"Event {i + 1}" for i, name in enumerate(names)},
        "thresholds": {name: float(thresholds.get(old, 0.5))
                       for old, name in zip(previous, names)},
        "fps": float(meta["fps"]),
        "num_frames": int(meta["num_frames"]),
        "img_size": int(meta["img_size"]),
    }

In [ ]:
%%writefile ved/model.py
"""VideoMAEv2 ViT backbone (compatible with the official OpenGVLab checkpoints)
plus a small multi-label classification head.

The official distilled checkpoint `distill/vit_s_k710_dl_from_giant.pth`
(ViT-Small, 16 frames, 224px, tubelet 2, patch 16) loads into this module.
The architecture is inferred from the checkpoint itself, so ViT-Base
checkpoints also work.
"""
from __future__ import annotations

import math
from functools import partial

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .labels import neutral_metadata


# --------------------------------------------------------------------------- #
# Building blocks
# --------------------------------------------------------------------------- #
class DropPath(nn.Module):
    def __init__(self, p: float = 0.0):
        super().__init__()
        self.p = p

    def forward(self, x):
        if self.p == 0.0 or not self.training:
            return x
        keep = 1 - self.p
        mask = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep)
        return x * mask / keep


class Mlp(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class Attention(nn.Module):
    def __init__(self, dim, num_heads, qkv_bias=True):
        super().__init__()
        self.num_heads = num_heads
        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        if qkv_bias:
            self.q_bias = nn.Parameter(torch.zeros(dim))
            self.v_bias = nn.Parameter(torch.zeros(dim))
        else:
            self.q_bias = self.v_bias = None
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        bias = None
        if self.q_bias is not None:
            bias = torch.cat([self.q_bias, torch.zeros_like(self.v_bias), self.v_bias])
        qkv = F.linear(x, self.qkv.weight, bias)
        qkv = qkv.reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        x = F.scaled_dot_product_attention(q, k, v)
        x = x.transpose(1, 2).reshape(B, N, C)
        return self.proj(x)


class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, drop_path=0.0, init_values=0.0):
        super().__init__()
        norm = partial(nn.LayerNorm, eps=1e-6)
        self.norm1 = norm(dim)
        self.attn = Attention(dim, num_heads)
        self.drop_path = DropPath(drop_path)
        self.norm2 = norm(dim)
        self.mlp = Mlp(dim, int(dim * mlp_ratio))
        if init_values > 0:
            self.gamma_1 = nn.Parameter(init_values * torch.ones(dim))
            self.gamma_2 = nn.Parameter(init_values * torch.ones(dim))
        else:
            self.gamma_1 = self.gamma_2 = None

    def forward(self, x):
        if self.gamma_1 is None:
            x = x + self.drop_path(self.attn(self.norm1(x)))
            x = x + self.drop_path(self.mlp(self.norm2(x)))
        else:
            x = x + self.drop_path(self.gamma_1 * self.attn(self.norm1(x)))
            x = x + self.drop_path(self.gamma_2 * self.mlp(self.norm2(x)))
        return x


def sinusoid_table(n_position, d_hid):
    def angle_vec(pos):
        return [pos / np.power(10000, 2 * (j // 2) / d_hid) for j in range(d_hid)]

    table = np.array([angle_vec(p) for p in range(n_position)])
    table[:, 0::2] = np.sin(table[:, 0::2])
    table[:, 1::2] = np.cos(table[:, 1::2])
    return torch.tensor(table, dtype=torch.float32).unsqueeze(0)


class VideoViT(nn.Module):
    """VideoMAEv2 encoder with mean pooling. Input: (B, 3, T, H, W)."""

    def __init__(self, img_size=224, patch_size=16, tubelet_size=2, num_frames=16,
                 embed_dim=384, depth=12, num_heads=6, mlp_ratio=4.0,
                 drop_path_rate=0.1, init_values=0.0):
        super().__init__()
        self.embed_dim = embed_dim
        self.depth = depth
        self.patch_embed = nn.Module()
        self.patch_embed.proj = nn.Conv3d(
            3, embed_dim,
            kernel_size=(tubelet_size, patch_size, patch_size),
            stride=(tubelet_size, patch_size, patch_size))
        n_patches = (img_size // patch_size) ** 2 * (num_frames // tubelet_size)
        self.register_buffer("pos_embed", sinusoid_table(n_patches, embed_dim), persistent=True)
        dpr = torch.linspace(0, drop_path_rate, depth).tolist()
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio, dpr[i], init_values) for i in range(depth)])
        self.fc_norm = nn.LayerNorm(embed_dim, eps=1e-6)

    def forward(self, x):
        x = self.patch_embed.proj(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed.to(x.dtype)
        for blk in self.blocks:
            x = blk(x)
        return self.fc_norm(x.mean(1))


class EventClassifier(nn.Module):
    """Backbone + multi-label head (one sigmoid output per event class)."""

    def __init__(self, backbone_cfg: dict, num_classes: int, head_dropout: float = 0.3):
        super().__init__()
        self.backbone_cfg = dict(backbone_cfg)
        self.backbone = VideoViT(**backbone_cfg)
        self.dropout = nn.Dropout(head_dropout)
        self.head = nn.Linear(self.backbone.embed_dim, num_classes)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        return self.head(self.dropout(self.backbone(x)))


# --------------------------------------------------------------------------- #
# Checkpoint helpers
# --------------------------------------------------------------------------- #
def _clean_state_dict(ckpt):
    for key in ("model", "module", "state_dict"):
        if isinstance(ckpt, dict) and key in ckpt and isinstance(ckpt[key], dict):
            ckpt = ckpt[key]
    out = {}
    for k, v in ckpt.items():
        for prefix in ("module.", "backbone.", "encoder."):
            if k.startswith(prefix):
                k = k[len(prefix):]
        out[k] = v
    # checkpoints without mean pooling store the final norm as `norm.*`
    if "fc_norm.weight" not in out and "norm.weight" in out:
        out["fc_norm.weight"] = out.pop("norm.weight")
        out["fc_norm.bias"] = out.pop("norm.bias")
    return out


def infer_backbone_cfg(sd: dict, num_frames=16, img_size=224, drop_path_rate=0.1) -> dict:
    w = sd["patch_embed.proj.weight"]  # (D, 3, t, p, p)
    embed_dim, _, tubelet, patch, _ = w.shape
    depth = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))
    mlp_hidden = sd["blocks.0.mlp.fc1.weight"].shape[0]
    return dict(
        img_size=img_size, patch_size=int(patch), tubelet_size=int(tubelet),
        num_frames=num_frames, embed_dim=int(embed_dim), depth=int(depth),
        num_heads=int(embed_dim) // 64, mlp_ratio=mlp_hidden / embed_dim,
        drop_path_rate=drop_path_rate,
        init_values=0.1 if "blocks.0.gamma_1" in sd else 0.0,
    )


def build_from_pretrained(path: str, num_classes: int, num_frames=16, img_size=224,
                          drop_path_rate=0.1, head_dropout=0.3) -> EventClassifier:
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    sd = _clean_state_dict(ckpt)
    cfg = infer_backbone_cfg(sd, num_frames, img_size, drop_path_rate)
    model = EventClassifier(cfg, num_classes, head_dropout)
    bb_sd = {k: v for k, v in sd.items() if not k.startswith("head.")}
    if "pos_embed" in bb_sd and bb_sd["pos_embed"].shape != model.backbone.pos_embed.shape:
        bb_sd.pop("pos_embed")
    missing, unexpected = model.backbone.load_state_dict(bb_sd, strict=False)
    missing = [k for k in missing if k != "pos_embed"]
    print(f"[pretrained] backbone cfg: {cfg}")
    print(f"[pretrained] missing keys: {missing}")
    print(f"[pretrained] unexpected keys: {unexpected}")
    if any(k.startswith("blocks.") or k.startswith("patch_embed") for k in missing):
        raise RuntimeError("Backbone weights did not load correctly - check the checkpoint.")
    return model


def save_export(model: EventClassifier, path: str, meta: dict):
    sd = {k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()}
    cfg = dict(model.backbone_cfg)
    cfg["drop_path_rate"] = 0.0
    torch.save({"state_dict": sd, "backbone_cfg": cfg,
                "num_classes": model.head.out_features,
                "meta": neutral_metadata(meta, model.head.out_features)}, path)


def load_export(path: str, device="cpu"):
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    model = EventClassifier(ckpt["backbone_cfg"], ckpt["num_classes"], head_dropout=0.0)
    model.load_state_dict({k: v.float() for k, v in ckpt["state_dict"].items()})
    return model.to(device).eval(), neutral_metadata(ckpt["meta"], ckpt["num_classes"])

In [ ]:
%%writefile ved/video_io.py
"""Video decoding: sample frames at a fixed FPS and resize them to a square."""
from __future__ import annotations

from typing import Iterator, Tuple

import cv2
import numpy as np

VIDEO_EXTS = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".m4v", ".wmv", ".flv", ".mpg", ".mpeg", ".ts"}


def video_info(path: str) -> Tuple[float, int, float]:
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    if fps <= 1e-3 or fps > 240:
        fps = 30.0
    return fps, n, (n / fps if n > 0 else 0.0)


def iter_frames(path: str, target_fps: float, size: int) -> Iterator[np.ndarray]:
    """Yield RGB uint8 frames (size x size x 3) sampled at `target_fps`.

    Frame k corresponds to timestamp k / target_fps seconds.
    Frames are squashed (not cropped) to keep the whole scene visible.
    """
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {path}")
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    if src_fps <= 1e-3 or src_fps > 240:
        src_fps = 30.0
    step = 1.0 / target_fps
    next_t = 0.0
    idx = 0
    while True:
        ok = cap.grab()
        if not ok:
            break
        t = idx / src_fps
        idx += 1
        if t + 1e-6 < next_t:
            continue
        ok, frame = cap.retrieve()
        if not ok:
            break
        frame = cv2.resize(frame, (size, size), interpolation=cv2.INTER_AREA)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # emit the same frame several times if the source FPS < target FPS
        while next_t <= t + 1e-6:
            yield frame
            next_t += step
    cap.release()


def read_frames(path: str, target_fps: float, size: int) -> np.ndarray:
    frames = list(iter_frames(path, target_fps, size))
    if not frames:
        raise IOError(f"No frames decoded from: {path}")
    return np.stack(frames)

In [ ]:
%%writefile ved/postprocess.py
"""Turn sliding-window probabilities into merged time segments.

Pipeline (per class):
  1. Overlap-add: every frame gets the mean (or max) probability of all
     windows that cover it  -> a frame-level score curve.
  2. Temporal smoothing (moving average) to remove single-window spikes.
  3. Hysteresis thresholding: a segment is a connected region where
     score >= low that contains at least one frame with score >= high.
     This keeps boundaries stable without flickering on/off.
  4. Gap merging: segments of the same class separated by <= merge_gap
     seconds are joined (the event was briefly occluded / low score).
  5. Minimum duration filter: drop segments shorter than min_dur seconds.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Dict, List, Sequence

import numpy as np


@dataclass
class Segment:
    label: str
    start: float
    end: float
    peak: float
    mean: float

    @property
    def duration(self):
        return self.end - self.start

    def to_dict(self):
        d = asdict(self)
        d["duration"] = round(self.duration, 3)
        return d


def windows_to_frame_scores(starts: Sequence[int], probs: np.ndarray, num_frames_total: int,
                            window_len: int, agg: str = "mean") -> np.ndarray:
    """starts: window start frame indices; probs: (num_windows, C)."""
    C = probs.shape[1]
    T = max(num_frames_total, 1)
    if agg == "max":
        out = np.zeros((T, C), dtype=np.float32)
        for s, p in zip(starts, probs):
            e = min(s + window_len, T)
            out[s:e] = np.maximum(out[s:e], p)
        return out
    acc = np.zeros((T, C), dtype=np.float64)
    cnt = np.zeros((T, 1), dtype=np.float64)
    for s, p in zip(starts, probs):
        e = min(s + window_len, T)
        acc[s:e] += p
        cnt[s:e] += 1
    cnt[cnt == 0] = 1
    return (acc / cnt).astype(np.float32)


def smooth(scores: np.ndarray, k: int) -> np.ndarray:
    if k <= 1:
        return scores
    kernel = np.ones(k) / k
    pad = k // 2
    out = np.empty_like(scores)
    for c in range(scores.shape[1]):
        x = np.pad(scores[:, c], (pad, k - 1 - pad), mode="edge")
        out[:, c] = np.convolve(x, kernel, mode="valid")
    return out


def hysteresis_regions(s: np.ndarray, high: float, low: float):
    above = s >= low
    regions, i, n = [], 0, len(s)
    while i < n:
        if above[i]:
            j = i
            while j < n and above[j]:
                j += 1
            if s[i:j].max() >= high:
                regions.append((i, j))
            i = j
        else:
            i += 1
    return regions


def merge_regions(regions, max_gap_frames: int):
    if not regions:
        return []
    merged = [list(regions[0])]
    for s, e in regions[1:]:
        if s - merged[-1][1] <= max_gap_frames:
            merged[-1][1] = e
        else:
            merged.append([s, e])
    return [tuple(r) for r in merged]


def frame_scores_to_segments(scores: np.ndarray, fps: float, class_names: List[str],
                             thresholds: Dict[str, float], low_ratio: float = 0.6,
                             smooth_sec: float = 1.0, merge_gap: float = 2.0,
                             min_dur: float = 1.0) -> List[Segment]:
    sm = smooth(scores, max(1, int(round(smooth_sec * fps))))
    segs: List[Segment] = []
    for c, name in enumerate(class_names):
        high = float(thresholds.get(name, 0.5))
        low = high * low_ratio
        regions = hysteresis_regions(sm[:, c], high, low)
        regions = merge_regions(regions, int(round(merge_gap * fps)))
        for s, e in regions:
            if (e - s) / fps < min_dur:
                continue
            seg_scores = sm[s:e, c]
            segs.append(Segment(name, round(s / fps, 2), round(e / fps, 2),
                                round(float(seg_scores.max()), 3), round(float(seg_scores.mean()), 3)))
    segs.sort(key=lambda x: (x.start, x.label))
    return segs


def fmt_time(sec: float) -> str:
    m, s = divmod(sec, 60)
    h, m = divmod(int(m), 60)
    return f"{h:02d}:{m:02d}:{s:05.2f}" if h else f"{m:02d}:{s:05.2f}"


def format_segments(segs: List[Segment], display: Dict[str, str] | None = None) -> str:
    if not segs:
        return "No events detected."
    lines = []
    for s in segs:
        name = (display or {}).get(s.label, s.label)
        lines.append(f"[{fmt_time(s.start)} -> {fmt_time(s.end)}] Detected {name} "
                     f"from {s.start:.2f}s until {s.end:.2f}s "
                     f"(duration {s.duration:.2f}s, peak {s.peak:.2f}, mean {s.mean:.2f})")
    return "\n".join(lines)

In [ ]:
%%writefile ved/inference.py
"""Sliding-window inference over long videos (runs fine on an RTX 4070, 12 GB)."""
from __future__ import annotations

import json
import math
import time
from collections import deque
from typing import Dict, Optional

import numpy as np
import torch

from .model import load_export
from .postprocess import format_segments, frame_scores_to_segments, windows_to_frame_scores
from .video_io import iter_frames, video_info

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1, 1)


def to_model_input(batch_uint8: torch.Tensor) -> torch.Tensor:
    """(B, T, H, W, 3) uint8 -> (B, 3, T, H, W) normalized float."""
    x = batch_uint8.permute(0, 4, 1, 2, 3).float().div_(255.0)
    return (x - MEAN.to(x.device)) / STD.to(x.device)


class Detector:
    def __init__(self, ckpt_path: str, device: Optional[str] = None, fp16: bool = True):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model, self.meta = load_export(ckpt_path, self.device)
        self.fp16 = fp16 and self.device.startswith("cuda")
        if self.fp16:
            self.model.half()
        self.class_names = self.meta["class_names"]
        self.display = self.meta.get("display_names", {})
        self.fps = float(self.meta["fps"])
        self.win = int(self.meta["num_frames"])
        self.size = int(self.meta["img_size"])
        self.thresholds = dict(self.meta.get("thresholds", {}))

    @torch.no_grad()
    def _run(self, windows):
        x = torch.from_numpy(np.stack(windows)).to(self.device, non_blocking=True)
        x = to_model_input(x)
        if self.fp16:
            x = x.half()
        return torch.sigmoid(self.model(x).float()).cpu().numpy()

    def predict_windows(self, video_path: str, stride_sec: float = 0.5, batch_size: int = 8,
                        verbose: bool = True):
        if batch_size < 1:
            raise ValueError("batch_size must be at least 1.")
        if not math.isfinite(stride_sec) or not 0 < stride_sec <= self.win / self.fps:
            raise ValueError("stride_sec must be positive and no longer than one model window.")
        stride = max(1, int(round(stride_sec * self.fps)))
        buf = deque(maxlen=self.win)
        pending, pending_starts = [], []
        starts, probs = [], []
        n = 0
        _, _, dur = video_info(video_path)
        t0 = time.time()
        last_start = -1
        for frame in iter_frames(video_path, self.fps, self.size):
            buf.append(frame)
            n += 1
            s = n - self.win
            if s >= 0 and s % stride == 0:
                pending.append(np.stack(buf)); pending_starts.append(s); last_start = s
            if len(pending) >= batch_size:
                probs.append(self._run(pending)); starts += pending_starts
                pending, pending_starts = [], []
                if verbose and dur > 0:
                    print(f"\r  processed {n / self.fps:7.1f}s / {dur:.1f}s "
                          f"({(n / self.fps) / max(time.time() - t0, 1e-6):.1f}x realtime)",
                          end="", flush=True)
        if n == 0:
            raise IOError(f"No frames decoded from {video_path}")
        if n < self.win:  # very short video: pad by repeating the last frame
            frames = list(buf) + [buf[-1]] * (self.win - n)
            pending.append(np.stack(frames)); pending_starts.append(0)
        elif last_start != n - self.win:  # make sure the tail is covered
            pending.append(np.stack(buf)); pending_starts.append(n - self.win)
        if pending:
            probs.append(self._run(pending)); starts += pending_starts
        if verbose:
            print(f"\r  done: {n / self.fps:.1f}s of video in {time.time() - t0:.1f}s" + " " * 20)
        return np.array(starts), np.concatenate(probs, 0), n

    def detect(self, video_path: str, stride_sec: float = 0.5, agg: str = "mean",
               thresholds: Optional[Dict[str, float]] = None, low_ratio: float = 0.6,
               smooth_sec: float = 1.0, merge_gap: float = 2.0, min_dur: float = 1.0,
               batch_size: int = 8, verbose: bool = True):
        if thresholds:
            if set(thresholds) - set(self.class_names):
                raise ValueError("Threshold keys must use event_1, event_2, etc.")
            if any(not math.isfinite(v) or not 0 <= v <= 1 for v in thresholds.values()):
                raise ValueError("Thresholds must be finite numbers between 0 and 1.")
        starts, probs, n = self.predict_windows(video_path, stride_sec, batch_size, verbose)
        frame_scores = windows_to_frame_scores(starts, probs, n, self.win, agg)
        thr = dict(self.thresholds)
        thr.update(thresholds or {})
        segs = frame_scores_to_segments(frame_scores, self.fps, self.class_names, thr,
                                        low_ratio, smooth_sec, merge_gap, min_dur)
        return {"video": video_path, "segments": segs, "frame_scores": frame_scores,
                "fps": self.fps, "thresholds": thr}

    def report(self, result) -> str:
        return format_segments(result["segments"], self.display)


def save_result(result, json_path: str, display: Optional[Dict[str, str]] = None):
    out = {"video": result["video"], "thresholds": result["thresholds"],
           "segments": [dict(s.to_dict(), display_name=(display or {}).get(s.label, s.label))
                        for s in result["segments"]]}
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)


def plot_timeline(result, class_names, path: Optional[str] = None, display=None):
    import matplotlib.pyplot as plt

    sc = result["frame_scores"]
    t = np.arange(len(sc)) / result["fps"]
    fig, axes = plt.subplots(len(class_names), 1, figsize=(14, 2.4 * len(class_names)), sharex=True)
    axes = np.atleast_1d(axes)
    for c, (ax, name) in enumerate(zip(axes, class_names)):
        ax.plot(t, sc[:, c], lw=0.8)
        thr = result["thresholds"].get(name, 0.5)
        ax.axhline(thr, color="r", ls="--", lw=0.8)
        for s in result["segments"]:
            if s.label == name:
                ax.axvspan(s.start, s.end, color="orange", alpha=0.3)
        ax.set_ylim(0, 1)
        ax.set_ylabel((display or {}).get(name, name))
    axes[-1].set_xlabel("time (s)")
    plt.tight_layout()
    if path:
        plt.savefig(path, dpi=120)
    return fig

In [ ]:
%%writefile ved/train.py
"""Dataset scanning, frame caching, MIL training and cross-validation."""
from __future__ import annotations

import copy
import hashlib
import math
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from .inference import MEAN, STD
from .model import build_from_pretrained
from .video_io import VIDEO_EXTS, read_frames


# --------------------------------------------------------------------------- #
# Dataset discovery
# --------------------------------------------------------------------------- #
def find_dataset_root(base: str) -> str:
    """Return the first directory under `base` that has `positive` and `negative` subfolders."""
    for root, dirs, _ in os.walk(base):
        low = {d.lower() for d in dirs}
        if "positive" in low and "negative" in low:
            return root
    raise FileNotFoundError(f"No folder with 'positive' and 'negative' subfolders under {base}")


def _videos_in(folder: str) -> List[str]:
    out = []
    for root, _, files in os.walk(folder):
        for f in files:
            if os.path.splitext(f)[1].lower() in VIDEO_EXTS:
                out.append(os.path.join(root, f))
    return sorted(out)


def scan_dataset(root: str, folder_to_class: Dict[str, str], class_names: List[str],
                 group_regex: Optional[str] = None) -> List[dict]:
    """positive/<sub>/ -> class from folder_to_class; negative/** -> all-zero label."""
    subdirs = {d.lower(): os.path.join(root, d) for d in os.listdir(root)}
    items = []
    pos_dir, neg_dir = subdirs["positive"], subdirs["negative"]
    for sub in sorted(os.listdir(pos_dir)):
        full = os.path.join(pos_dir, sub)
        if not os.path.isdir(full):
            continue
        cls = folder_to_class.get(sub.lower())
        if cls is None:
            print(f"[warn] unknown positive subfolder '{sub}' - skipped")
            continue
        y = np.zeros(len(class_names), np.float32)
        y[class_names.index(cls)] = 1
        for p in _videos_in(full):
            items.append(dict(path=p, y=y.copy(), source=f"positive/{sub}"))
    for p in _videos_in(neg_dir):
        rel = os.path.relpath(p, neg_dir).split(os.sep)[0]
        items.append(dict(path=p, y=np.zeros(len(class_names), np.float32), source=f"negative/{rel}"))
    for it in items:
        name = os.path.splitext(os.path.basename(it["path"]))[0]
        m = re.search(group_regex, name) if group_regex else None
        it["group"] = m.group(1) if m else it["path"]
    return items


def summarize(items, class_names):
    from collections import Counter
    c = Counter(it["source"] for it in items)
    for k in sorted(c):
        print(f"  {k:25s} {c[k]:5d}")
    Y = np.stack([it["y"] for it in items])
    for i, n in enumerate(class_names):
        print(f"  -> {n:20s} positives: {int(Y[:, i].sum())}")
    print(f"  -> negatives (all zero): {int((Y.sum(1) == 0).sum())}   total: {len(items)}")
    print(f"  -> unique groups: {len({it['group'] for it in items})}")


# --------------------------------------------------------------------------- #
# Frame cache (decode each clip once -> uint8 .npy)
# --------------------------------------------------------------------------- #
def build_cache(items, cache_dir, fps, size, workers=8):
    os.makedirs(cache_dir, exist_ok=True)

    def work(it):
        key = hashlib.md5(f"{it['path']}|{fps}|{size}".encode()).hexdigest()
        out = os.path.join(cache_dir, key + ".npy")
        if not os.path.exists(out):
            try:
                arr = read_frames(it["path"], fps, size)
            except Exception as e:  # noqa
                return it, None, str(e)
            np.save(out, arr)
        return it, out, None

    ok, bad = [], []
    t0 = time.time()
    with ThreadPoolExecutor(workers) as ex:
        for i, (it, out, err) in enumerate(ex.map(work, items)):
            if out is None:
                bad.append((it["path"], err))
            else:
                it["cache"] = out
                it["n_frames"] = int(np.load(out, mmap_mode="r").shape[0])
                ok.append(it)
            if (i + 1) % 25 == 0 or i + 1 == len(items):
                print(f"\r  cached {i + 1}/{len(items)} ({time.time() - t0:.0f}s)", end="", flush=True)
    print()
    for p, e in bad:
        print(f"[warn] failed to decode {p}: {e}")
    return ok


# --------------------------------------------------------------------------- #
# Dataset: K windows per clip (multiple-instance learning)
# --------------------------------------------------------------------------- #
class ClipDataset(Dataset):
    def __init__(self, items, num_frames, k_windows, train):
        self.items, self.T, self.K, self.train = items, num_frames, k_windows, train

    def __len__(self):
        return len(self.items)

    def _starts(self, n):
        max_s = max(n - self.T, 0)
        if self.train:
            edges = np.linspace(0, max_s, self.K + 1)
            return [int(np.random.randint(int(edges[i]), max(int(edges[i + 1]), int(edges[i]) + 1)))
                    for i in range(self.K)]
        return [int(round(v)) for v in np.linspace(0, max_s, self.K)]

    def __getitem__(self, i):
        it = self.items[i]
        arr = np.load(it["cache"], mmap_mode="r")
        n = arr.shape[0]
        wins = []
        for s in self._starts(n):
            idx = np.clip(np.arange(s, s + self.T), 0, n - 1)
            wins.append(np.asarray(arr[idx]))
        return torch.from_numpy(np.stack(wins)), torch.from_numpy(it["y"])


def gpu_prepare(x_uint8, size, train):
    """(B, K, T, H, W, 3) uint8 -> (B*K, 3, T, size, size) normalized, with augmentation."""
    B, K, T, H, W, _ = x_uint8.shape
    x = x_uint8.view(B * K, T, H, W, 3).permute(0, 1, 4, 2, 3).float().div_(255.0)  # N,T,3,H,W
    if train:
        out = []
        for v in x:
            scale = np.random.uniform(0.6, 1.0)
            ratio = math.exp(np.random.uniform(math.log(3 / 4), math.log(4 / 3)))
            h = min(H, int(round(math.sqrt(scale * H * W / ratio))))
            w = min(W, int(round(math.sqrt(scale * H * W * ratio))))
            y0, x0 = np.random.randint(0, H - h + 1), np.random.randint(0, W - w + 1)
            v = v[:, :, y0:y0 + h, x0:x0 + w]
            v = F.interpolate(v, size=(size, size), mode="bilinear", align_corners=False)
            if np.random.rand() < 0.5:
                v = v.flip(-1)
            c = np.random.uniform(0.75, 1.25)
            b = np.random.uniform(0.75, 1.25)
            m = v.mean()
            v = ((v - m) * c + m) * b
            out.append(v.clamp_(0, 1))
        x = torch.stack(out)
    elif H != size or W != size:
        x = F.interpolate(x.flatten(0, 1), size=(size, size), mode="bilinear",
                          align_corners=False).view(B * K, T, 3, size, size)
    x = x.permute(0, 2, 1, 3, 4)  # N,3,T,H,W
    return (x - MEAN.to(x.device)) / STD.to(x.device)


def lse_pool(logits, r=5.0):
    """Smooth max over windows: (B, K, C) -> (B, C)."""
    K = logits.shape[1]
    return (torch.logsumexp(r * logits, dim=1) - math.log(K)) / r


# --------------------------------------------------------------------------- #
# Optimizer with layer-wise LR decay + partial freezing
# --------------------------------------------------------------------------- #
def make_optimizer(model, cfg):
    depth = model.backbone.depth
    groups = {}
    for n, p in model.named_parameters():
        if n.startswith("backbone.patch_embed"):
            lid = 0
        elif n.startswith("backbone.blocks."):
            lid = int(n.split(".")[2]) + 1
        elif n.startswith("backbone."):
            lid = depth + 1
        else:
            lid = None  # head
        if lid is not None and cfg["freeze_blocks"] > 0 and lid <= cfg["freeze_blocks"]:
            p.requires_grad_(False)
            continue
        no_wd = p.ndim == 1 or n.endswith(".bias") or "q_bias" in n or "v_bias" in n
        if lid is None:
            lr = cfg["head_lr"]
        else:
            lr = cfg["backbone_lr"] * cfg["layer_decay"] ** (depth + 1 - lid)
        key = (lr, 0.0 if no_wd else cfg["weight_decay"])
        groups.setdefault(key, []).append(p)
    param_groups = [dict(params=ps, lr=lr, weight_decay=wd, base_lr=lr) for (lr, wd), ps in groups.items()]
    n_train = sum(p.numel() for g in param_groups for p in g["params"])
    print(f"  trainable params: {n_train / 1e6:.2f}M")
    return torch.optim.AdamW(param_groups, betas=(0.9, 0.999))


def lr_lambda(total, warmup):
    def f(step):
        if step < warmup:
            return (step + 1) / warmup
        prog = (step - warmup) / max(1, total - warmup)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * prog))
    return f


# --------------------------------------------------------------------------- #
# Metrics
# --------------------------------------------------------------------------- #
def clip_metrics(y_true, y_prob, class_names, thresholds=None):
    from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
    res = {}
    for c, name in enumerate(class_names):
        yt, yp = y_true[:, c], y_prob[:, c]
        r = {"AP": float(average_precision_score(yt, yp)) if yt.sum() > 0 else float("nan"),
             "AUC": float(roc_auc_score(yt, yp)) if 0 < yt.sum() < len(yt) else float("nan")}
        if thresholds is None:
            prec, rec, thr = precision_recall_curve(yt, yp)
            f1 = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
            best = int(np.nanargmax(f1[:-1])) if len(thr) else 0
            t = float(thr[best]) if len(thr) else 0.5
        else:
            t = float(thresholds[name])
        pred = yp >= t
        tp = int((pred & (yt == 1)).sum()); fp = int((pred & (yt == 0)).sum())
        fn = int((~pred & (yt == 1)).sum())
        p_ = tp / max(tp + fp, 1); r_ = tp / max(tp + fn, 1)
        r.update(threshold=t, precision=p_, recall=r_, F1=2 * p_ * r_ / max(p_ + r_, 1e-9),
                 TP=tp, FP=fp, FN=fn)
        res[name] = r
    res["mAP"] = float(np.nanmean([res[n]["AP"] for n in class_names]))
    return res


# --------------------------------------------------------------------------- #
# Training
# --------------------------------------------------------------------------- #
@torch.no_grad()
def predict_clips(model, items, cfg, device):
    ds = ClipDataset(items, cfg["num_frames"], cfg["k_eval"], train=False)
    dl = DataLoader(ds, batch_size=max(1, cfg["batch_size"] // 2), shuffle=False,
                    num_workers=cfg["num_workers"], pin_memory=True)
    model.eval()
    out = []
    for x, _ in dl:
        B, K = x.shape[:2]
        x = gpu_prepare(x.to(device, non_blocking=True), cfg["img_size"], train=False)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device == "cuda"):
            p = torch.sigmoid(model(x).float()).view(B, K, -1)
        out.append(p.max(1).values.cpu().numpy())  # clip score = best window
    return np.concatenate(out)


def train_model(cfg, pretrained_path, train_items, val_items=None, epochs=None, tag=""):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    C = len(cfg["class_names"])
    epochs = epochs or cfg["epochs"]
    model = build_from_pretrained(pretrained_path, C, cfg["num_frames"], cfg["img_size"],
                                  cfg["drop_path"], cfg["head_dropout"]).to(device)
    opt = make_optimizer(model, cfg)
    ds = ClipDataset(train_items, cfg["num_frames"], cfg["k_train"], train=True)
    dl = DataLoader(ds, batch_size=cfg["batch_size"], shuffle=True, drop_last=True,
                    num_workers=cfg["num_workers"], pin_memory=True, persistent_workers=cfg["num_workers"] > 0)
    total = epochs * len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda(total, cfg["warmup_epochs"] * len(dl)))
    Y = np.stack([it["y"] for it in train_items])
    pos = Y.sum(0); neg = len(Y) - pos
    pos_weight = torch.tensor(np.sqrt(neg / np.clip(pos, 1, None)), dtype=torch.float32, device=device)
    print(f"  {tag} train={len(train_items)} val={len(val_items) if val_items else 0} "
          f"epochs={epochs} pos_weight={pos_weight.cpu().numpy().round(2)}")

    history, best = [], dict(score=-1, state=None, epoch=0, probs=None)
    for ep in range(1, epochs + 1):
        model.train()
        t0, losses = time.time(), []
        for x, y in dl:
            B, K = x.shape[:2]
            x = gpu_prepare(x.to(device, non_blocking=True), cfg["img_size"], train=True)
            y = y.to(device)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device == "cuda"):
                logits = model(x).float().view(B, K, -1)
            clip_logits = lse_pool(logits, cfg["lse_r"])
            loss = F.binary_cross_entropy_with_logits(clip_logits, y, pos_weight=pos_weight)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step(); sched.step()
            losses.append(loss.item())
        rec = dict(epoch=ep, loss=float(np.mean(losses)), time=time.time() - t0)
        if val_items:
            probs = predict_clips(model, val_items, cfg, device)
            yv = np.stack([it["y"] for it in val_items])
            m = clip_metrics(yv, probs, cfg["class_names"])
            rec["val_mAP"] = m["mAP"]
            for n in cfg["class_names"]:
                rec[f"AP_{n}"] = m[n]["AP"]
            if m["mAP"] > best["score"]:
                best.update(score=m["mAP"], epoch=ep, probs=probs,
                            state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        history.append(rec)
        print("  " + tag + " " + "  ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}"
                                          for k, v in rec.items()))
    if best["state"] is not None:
        model.load_state_dict(best["state"])
    return model, history, best


def make_folds(items, n_splits, seed=42):
    from sklearn.model_selection import StratifiedGroupKFold
    Y = np.stack([it["y"] for it in items])
    strat = (Y * (2 ** np.arange(Y.shape[1]))).sum(1).astype(int)
    groups = [it["group"] for it in items]
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(sgkf.split(np.zeros(len(items)), strat, groups))

In [ ]:
%%writefile infer.py
"""Command-line inference (designed for a consumer GPU such as an RTX 4070).

Example:
    python infer.py --ckpt export/model.pt --video my_video.mp4 --plot
"""
import argparse
import os

from ved.inference import Detector, plot_timeline, save_result


def main():
    ap = argparse.ArgumentParser(description="Detect and localize events in a video.")
    ap.add_argument("--ckpt", required=True, help="exported model.pt")
    ap.add_argument("--video", required=True, nargs="+", help="one or more video files")
    ap.add_argument("--stride", type=float, default=0.5, help="window stride in seconds")
    ap.add_argument("--agg", choices=["mean", "max"], default="mean",
                    help="how overlapping windows are combined per frame")
    ap.add_argument("--smooth", type=float, default=1.0, help="moving-average length (s)")
    ap.add_argument("--merge-gap", type=float, default=2.0, help="merge segments closer than this (s)")
    ap.add_argument("--min-dur", type=float, default=1.0, help="drop segments shorter than this (s)")
    ap.add_argument("--low-ratio", type=float, default=0.6, help="hysteresis low = high * ratio")
    ap.add_argument("--thr", nargs="*", default=[], help="override thresholds, e.g. event_1=0.6")
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--device", choices=["cuda", "cpu"], default=None,
                    help="default: CUDA when available, otherwise CPU")
    ap.add_argument("--fp32", action="store_true", help="disable fp16")
    ap.add_argument("--out-dir", default="results")
    ap.add_argument("--plot", action="store_true", help="save a score timeline PNG")
    args = ap.parse_args()

    try:
        thr = {k: float(v) for k, v in (t.split("=", 1) for t in args.thr)}
    except ValueError:
        ap.error("--thr must use event_1=0.6 event_2=0.5 format")
    det = Detector(args.ckpt, device=args.device, fp16=not args.fp32)
    print(f"device={det.device}  classes={det.class_names}  thresholds={det.thresholds | thr}")
    os.makedirs(args.out_dir, exist_ok=True)
    for video in args.video:
        print(f"\n=== {video}")
        res = det.detect(video, stride_sec=args.stride, agg=args.agg, thresholds=thr,
                         low_ratio=args.low_ratio, smooth_sec=args.smooth,
                         merge_gap=args.merge_gap, min_dur=args.min_dur, batch_size=args.batch)
        print(det.report(res))
        stem = os.path.splitext(os.path.basename(video))[0]
        save_result(res, os.path.join(args.out_dir, stem + ".json"), det.display)
        if args.plot:
            plot_timeline(res, det.class_names, os.path.join(args.out_dir, stem + ".png"), det.display)


if __name__ == "__main__":
    main()

## 2. Configuration

In [ ]:
# ---- data -----------------------------------------------------------------
DATA_SOURCE = "drive_mount"   # "gdown_folder" | "drive_mount" | "gdown_zip"
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/17uHSFRmn8QXLMbj5Tq8-qf474_DOd95O?usp=drive_link"
DRIVE_MOUNT_PATH = "/content/drive/MyDrive/dataset"   # used if DATA_SOURCE == "drive_mount"
ZIP_URL = ""                                          # used if DATA_SOURCE == "gdown_zip"
DATA_DIR = os.path.join(WORK, "data")
CACHE_DIR = os.path.join(WORK, "cache")
OUT_DIR = os.path.join(WORK, "export")

# If clips were cut from the same source video, give them a shared group so they
# never end up in both train and validation. Regex with ONE capture group applied to
# the file name, e.g. r"^(.*)_clip\d+$". None = every clip is its own group.
GROUP_REGEX = None

# ---- pretrained model ------------------------------------------------------
HF_REPO = "OpenGVLab/VideoMAE2"
HF_FILE = "distill/vit_s_k710_dl_from_giant.pth"

# ---- training --------------------------------------------------------------
CFG = dict(
    class_names=["event_1", "event_2"],
    folder_to_class={"og": "event_1", "sq": "event_2"},
    display_names={"event_1": "Event 1", "event_2": "Event 2"},
    fps=8, num_frames=16, img_size=224,   # 16 frames @ 8 fps = 2 s window (matches K710 sampling)
    k_train=4, k_eval=7,                  # MIL windows per 8 s clip
    batch_size=8,                         # clips per step (x k_train windows)
    epochs=30, warmup_epochs=3,
    backbone_lr=1.5e-4, head_lr=1e-3, layer_decay=0.75, weight_decay=0.05,
    drop_path=0.1, head_dropout=0.3,
    freeze_blocks=4,                      # freeze patch-embed + first N of 12 blocks (0 = full fine-tune)
    lse_r=5.0, grad_clip=1.0, num_workers=4,
)
N_FOLDS = 5
RUN_CV = True          # set False to skip CV and only train the final model
SEED = 42

## 3. Download the dataset

`gdown` can only list **50 files per Drive folder**. If a subfolder has more than 50 clips
(og / sq have 80), use one of the alternatives:
* **`drive_mount`**: open the shared link -> *Add shortcut to Drive*, then set `DRIVE_MOUNT_PATH`.
* **`gdown_zip`**: zip the `dataset` folder, share the zip, set `ZIP_URL`.

The cell checks the counts and stops if a folder looks truncated.

In [ ]:
import gdown, glob, zipfile
from ved.video_io import VIDEO_EXTS

if DATA_SOURCE == "gdown_folder":
    gdown.download_folder(DRIVE_FOLDER_URL, output=DATA_DIR, quiet=True, remaining_ok=True)
    data_base = DATA_DIR
elif DATA_SOURCE == "drive_mount":
    from google.colab import drive
    drive.mount("/content/drive")
    data_base = DRIVE_MOUNT_PATH
elif DATA_SOURCE == "gdown_zip":
    os.makedirs(DATA_DIR, exist_ok=True)
    zpath = gdown.download(ZIP_URL, os.path.join(WORK, "dataset.zip"), quiet=False, fuzzy=True)
    zipfile.ZipFile(zpath).extractall(DATA_DIR)
    data_base = DATA_DIR
else:
    raise ValueError(DATA_SOURCE)

truncated = []
for root, dirs, files in os.walk(data_base):
    n = sum(os.path.splitext(f)[1].lower() in VIDEO_EXTS for f in files)
    if n:
        print(f"{n:5d}  {root}")
    if n == 50 and DATA_SOURCE == "gdown_folder":
        truncated.append(root)
if truncated:
    raise RuntimeError("These folders have exactly 50 files -> gdown hit its listing limit:\n"
                       + "\n".join(truncated) + "\nUse DATA_SOURCE='drive_mount' or 'gdown_zip'.")

## 4. Download the pretrained VideoMAEv2 ViT-S

The Hugging Face repo `OpenGVLab/VideoMAE2` may be **gated**: open
https://huggingface.co/OpenGVLab/VideoMAE2, accept the terms, create a read token and
store it as the Colab secret `HF_TOKEN` (key icon in the left sidebar) or env var `HF_TOKEN`.

In [ ]:
from huggingface_hub import hf_hub_download
token = os.environ.get("HF_TOKEN")
if token is None and IN_COLAB:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
PRETRAINED = hf_hub_download(HF_REPO, HF_FILE, token=token, local_dir=os.path.join(WORK, "pretrained"))
print(PRETRAINED, f"{os.path.getsize(PRETRAINED) / 1e6:.1f} MB")

# sanity check: the checkpoint loads into our backbone
from ved.model import build_from_pretrained
_ = build_from_pretrained(PRETRAINED, num_classes=2); del _

## 5. Scan dataset + cache frames

In [ ]:
import numpy as np, random, torch
from ved.train import find_dataset_root, scan_dataset, summarize, build_cache

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
root = find_dataset_root(data_base)
print("dataset root:", root)
items = scan_dataset(root, CFG["folder_to_class"], CFG["class_names"], GROUP_REGEX)
summarize(items, CFG["class_names"])
items = build_cache(items, CACHE_DIR, CFG["fps"], CFG["img_size"], workers=8)
lens = np.array([it["n_frames"] for it in items]) / CFG["fps"]
print(f"clip length: min {lens.min():.1f}s  median {np.median(lens):.1f}s  max {lens.max():.1f}s")

## 6. 5-fold cross-validation (for the thesis numbers + threshold calibration)

Each fold trains from the pretrained checkpoint. Out-of-fold (OOF) clip scores are
collected for every clip, then per-class AP / AUC / P / R / F1 are computed and the
F1-optimal threshold per class is stored for inference.

*Note:* the OOF predictions come from the best epoch on each validation fold, which is
slightly optimistic. For a strict number, keep a separate held-out test set.

In [ ]:
import json, pandas as pd
from ved.train import make_folds, train_model, clip_metrics

os.makedirs(OUT_DIR, exist_ok=True)
Y = np.stack([it["y"] for it in items])
thresholds = {n: 0.5 for n in CFG["class_names"]}
best_epochs, all_hist = [], []

if RUN_CV:
    oof = np.zeros_like(Y)
    for k, (tr, va) in enumerate(make_folds(items, N_FOLDS, SEED)):
        print(f"\n===== fold {k + 1}/{N_FOLDS} =====")
        model, hist, best = train_model(CFG, PRETRAINED, [items[i] for i in tr], [items[i] for i in va],
                                        tag=f"[f{k + 1}]")
        oof[va] = best["probs"]
        best_epochs.append(best["epoch"])
        for h in hist:
            all_hist.append(dict(h, fold=k + 1))
        del model; torch.cuda.empty_cache()

    metrics = clip_metrics(Y, oof, CFG["class_names"])
    thresholds = {n: metrics[n]["threshold"] for n in CFG["class_names"]}
    print("\n===== OOF clip-level metrics =====")
    print(pd.DataFrame({n: metrics[n] for n in CFG["class_names"]}).round(3))
    print("mAP:", round(metrics["mAP"], 4), " best epochs per fold:", best_epochs)
    neg = Y.sum(1) == 0
    for c, n in enumerate(CFG["class_names"]):
        print(f"false-positive rate on negative clips [{n}]: {(oof[neg, c] >= thresholds[n]).mean():.3f}")
    pd.DataFrame(all_hist).to_csv(os.path.join(OUT_DIR, "cv_history.csv"), index=False)
    pd.DataFrame(oof, columns=CFG["class_names"]).assign(
        path=[it["path"] for it in items], **{f"y_{n}": Y[:, i] for i, n in enumerate(CFG["class_names"])}
    ).to_csv(os.path.join(OUT_DIR, "oof_predictions.csv"), index=False)
    json.dump(metrics, open(os.path.join(OUT_DIR, "cv_metrics.json"), "w"), indent=2)

In [ ]:
# training curves + PR curves
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
if RUN_CV:
    h = pd.DataFrame(all_hist)
    fig, ax = plt.subplots(1, 3, figsize=(17, 4))
    for f, g in h.groupby("fold"):
        ax[0].plot(g.epoch, g.loss, label=f"fold {f}"); ax[1].plot(g.epoch, g.val_mAP, label=f"fold {f}")
    ax[0].set_title("train loss"); ax[1].set_title("val mAP"); ax[0].legend()
    for c, n in enumerate(CFG["class_names"]):
        p, r, _ = precision_recall_curve(Y[:, c], oof[:, c])
        ax[2].plot(r, p, label=f"{n} (AP {metrics[n]['AP']:.2f})")
    ax[2].set_xlabel("recall"); ax[2].set_ylabel("precision"); ax[2].set_title("OOF PR curve"); ax[2].legend()
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "cv_curves.png"), dpi=120); plt.show()

## 7. Train the final model on all clips and export

In [ ]:
from ved.model import save_export
final_epochs = int(round(np.mean(best_epochs))) if best_epochs else CFG["epochs"]
final_epochs = max(final_epochs, 10)
print("final training epochs:", final_epochs, " thresholds:", thresholds)
model, hist, _ = train_model(CFG, PRETRAINED, items, None, epochs=final_epochs, tag="[final]")
meta = dict(class_names=CFG["class_names"], display_names=CFG["display_names"], fps=CFG["fps"],
            num_frames=CFG["num_frames"], img_size=CFG["img_size"], thresholds=thresholds,
            pretrained=HF_FILE, train_clips=len(items), epochs=final_epochs, cfg=CFG)
ckpt_path = os.path.join(OUT_DIR, "model.pt")
save_export(model.cpu(), ckpt_path, meta)
print("saved", ckpt_path, f"{os.path.getsize(ckpt_path) / 1e6:.1f} MB")
del model; torch.cuda.empty_cache()

## 8. Inference on a long video

Set `TEST_VIDEO` to any video (upload it to Colab or put it on Drive). By default a
demo video is built by concatenating a few dataset clips so you can see the merging work.

Post-processing knobs: `stride_sec` (window hop), `smooth_sec`, `merge_gap` (join segments
closer than this), `min_dur`, `low_ratio` (hysteresis), per-class `thresholds`.

In [ ]:
from ved.inference import Detector, plot_timeline, save_result
TEST_VIDEO = ""   # e.g. "/content/drive/MyDrive/test_video.mp4"

if not TEST_VIDEO:
    import cv2
    rng = np.random.default_rng(0)
    pick = [items[i] for i in rng.choice(len(items), 6, replace=False)]
    TEST_VIDEO = os.path.join(WORK, "demo_concat.mp4")
    vw = cv2.VideoWriter(TEST_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), CFG["fps"], (CFG["img_size"],) * 2)
    t = 0.0
    for it in pick:
        arr = np.load(it["cache"])
        lab = [n for n, v in zip(CFG["class_names"], it["y"]) if v] or ["negative"]
        print(f"  {t:6.2f}s - {t + len(arr) / CFG['fps']:6.2f}s  ground truth clip label: {lab}")
        for fr in arr:
            vw.write(cv2.cvtColor(fr, cv2.COLOR_RGB2BGR))
        t += len(arr) / CFG["fps"]
    vw.release()
    print("(demo clips are from the training set, so this only checks the pipeline, not accuracy)\n")

det = Detector(ckpt_path)
res = det.detect(TEST_VIDEO, stride_sec=0.5, agg="mean", smooth_sec=1.0,
                 merge_gap=2.0, min_dur=1.0, low_ratio=0.6)
print(det.report(res))
save_result(res, os.path.join(OUT_DIR, "demo_result.json"), det.display)
plot_timeline(res, det.class_names, os.path.join(OUT_DIR, "demo_timeline.png"), det.display); plt.show()

## 9. Package the export for the RTX 4070 machine

In [ ]:
import shutil
pkg = os.path.join(WORK, "inference_package")
shutil.rmtree(pkg, ignore_errors=True)
os.makedirs(pkg)
shutil.copytree("ved", os.path.join(pkg, "ved"))
shutil.copy("infer.py", pkg)
for f in os.listdir(OUT_DIR):
    shutil.copy(os.path.join(OUT_DIR, f), pkg)
zip_path = shutil.make_archive(os.path.join(WORK, "inference_package"), "zip", pkg)
print("created", zip_path)
if IN_COLAB:
    from google.colab import drive, files
    try:
        drive.mount("/content/drive")
        shutil.copy(zip_path, "/content/drive/MyDrive/")
        print("copied to MyDrive/inference_package.zip")
    except Exception as e:
        print("Drive copy failed:", e); files.download(zip_path)